# Advanced Analytical Features

This notebook demonstrates the advanced symbolic analysis capabilities:

1. **Equilibrium Finding**: Disease-free and endemic equilibria
2. **Stability Analysis**: Jacobian matrices, eigenvalues, stability classification
3. **Sensitivity Analysis**: Parameter sensitivities, elasticity indices, perturbation analysis
4. **Parameter Ranking**: Identify most influential parameters

In [1]:
from epimodels.validation import SymbolicModel
from epimodels import BaseModel, ValidationError
from epimodels.validation.specs import ParameterSpec, VariableSpec, ModelConstraint
from collections import OrderedDict
from IPython.display import display, Markdown, Latex
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

print("✓ Imports successful")

✓ Imports successful


## 1. Create a Symbolic SIR Model

First, we'll create a symbolic representation of the SIR model.

In [2]:
# Create symbolic SIR model
sir = SymbolicModel()

# Add parameters
sir.add_parameter("beta", positive=True, real=True)
sir.add_parameter("gamma", positive=True, real=True)

# Add state variables
sir.add_variable("S", positive=True, real=True)
sir.add_variable("I", positive=True, real=True)
sir.add_variable("R", positive=True, real=True)

# Set total population
sir.set_total_population("N")

# Define ODEs
sir.define_ode("S", "-beta*S*I/N")
sir.define_ode("I", "beta*S*I/N - gamma*I")
sir.define_ode("R", "gamma*I")

print(f"Created SIR model with {len(sir.parameters)} parameters and {len(sir.variables)} variables")
print(f"Parameters: {list(sir.parameters.keys())}")
print(f"Variables: {list(sir.variables.keys())}")

Created SIR model with 2 parameters and 3 variables
Parameters: ['beta', 'gamma']
Variables: ['S', 'I', 'R']


## 2. Basic Reproduction Number (R0)

Compute R0 using the next-generation matrix method.

In [3]:
# Compute R0 symbolically
R0_symbolic = sir.compute_R0_next_generation()
print(f"R0 (symbolic): {R0_symbolic}")
display(Latex(fr"$(\LaTeX)$ $R_0 = {sir.to_latex(R0_symbolic)}$"))

# Evaluate numerically
params = {'beta': 0.3, 'gamma': 0.1, 'N': 1000}
R0_numeric = (sir.substitute_values(R0_symbolic, params)).evalf()
print(f"\nR0 (numeric, beta={params['beta']}, gamma={params['gamma']}): {R0_numeric}")

R0 (symbolic): -S*beta/(N*gamma)


<IPython.core.display.Latex object>


R0 (numeric, beta=0.3, gamma=0.1): -0.003*S


## 3. Equilibrium Analysis

Find all equilibrium points of the model.

In [8]:
# Parameter values
params = {'beta': 0.3, 'gamma': 0.1, 'N': 1000}

# Find all equilibria
print("Finding all equilibria...")
equilibria = sir.find_all_equilibria(params=params, numeric_fallback=True)

print(f"\nFound {len(equilibria)} equilibrium point(s):")
print("=" * 70)

for i, eq in enumerate(equilibria, 1):
    print(f"\nEquilibrium {i}: {eq['type'].upper()}")
    print(f"  Method: {eq['method']}")
    print(f"  Values:")
    for var in ['S', 'I', 'R']:
        val = eq.get(var, 0)
        if hasattr(val, 'evalf'):
            val = val.evalf()
        print(f"    {var} = {val}")

Finding all equilibria...


TypeError: cannot determine truth value of Relational: Abs(N - 1000.0) > 1.0e-6

### 3.1 Disease-Free Equilibrium (DFE)

In [ ]:
# Find DFE explicitly
dfe = sir.find_disease_free_equilibrium()
print("Disease-Free Equilibrium (DFE):")
print(f"  S* = {dfe.get('S', 'N')}")
print(f"  I* = {dfe.get('I', 0)}")
print(f"  R* = {dfe.get('R', 0)}")
print("\nInterpretation: Entire population is susceptible, no infection.")

### 3.2 Endemic Equilibrium

In [ ]:
# Find endemic equilibrium
print("Finding endemic equilibrium (R0 > 1)...")
endemic = sir.find_endemic_equilibrium(params)

if endemic:
    print("\nEndemic Equilibrium:")
    print(f"  Method: {endemic.get('method', 'unknown')}")
    for var in ['S', 'I', 'R']:
        val = endemic.get(var, 0)
        if hasattr(val, 'evalf'):
            val = float(val.evalf())
        print(f"  {var}* = {val:.2f}")
    
    print(f"\nInterpretation:")
    print(f"  - {endemic['I']:.0f} individuals remain infected at equilibrium")
    print(f"  - S* = N/R0 = {params['N']/R0_numeric:.2f}")
    print(f"  - Disease persists in the population")
else:
    print("No endemic equilibrium exists (R0 ≤ 1)")

### 3.3 Effect of R0 on Endemic Equilibrium

In [ ]:
# Test different R0 values
beta_values = [0.05, 0.1, 0.2, 0.3, 0.5]
gamma_fixed = 0.1
N_fixed = 1000

print("Effect of R0 on endemic equilibrium:")
print("=" * 70)
print(f"{'Beta':<10} {'R0':<10} {'S*':<15} {'I*':<15} {'Endemic?':<10}")
print("-" * 70)

for beta in beta_values:
    params_test = {'beta': beta, 'gamma': gamma_fixed, 'N': N_fixed}
    R0_test = beta / gamma_fixed
    endemic_test = sir.find_endemic_equilibrium(params_test)
    
    if endemic_test:
        S_star = float(endemic_test['S'].evalf()) if hasattr(endemic_test['S'], 'evalf') else endemic_test['S']
        I_star = float(endemic_test['I'].evalf()) if hasattr(endemic_test['I'], 'evalf') else endemic_test['I']
        print(f"{beta:<10.2f} {R0_test:<10.2f} {S_star:<15.2f} {I_star:<15.2f} {'Yes':<10}")
    else:
        print(f"{beta:<10.2f} {R0_test:<10.2f} {'N/A':<15} {'N/A':<15} {'No':<10}")

## 4. Stability Analysis

Analyze the stability of equilibrium points using Jacobian matrices and eigenvalues.

In [ ]:
# Compute Jacobian at DFE
J_dfe = sir.compute_jacobian(dfe, substitute_values=False)
print("Jacobian Matrix at DFE (symbolic):")
print(J_dfe)
print(f"\nShape: {J_dfe.shape}")

In [ ]:
# Compute eigenvalues symbolically
eigenvalues_sym = sir.compute_eigenvalues(J_dfe, numeric=False)
print("Eigenvalues (symbolic):")
for i, ev in enumerate(eigenvalues_sym, 1):
    print(f"  λ{i} = {ev}")

In [ ]:
# Compute eigenvalues numerically
J_dfe_numeric = sir.compute_jacobian(dfe, substitute_values=True)
eigenvalues_num = sir.compute_eigenvalues(J_dfe_numeric, numeric=True, params=params)

print("Eigenvalues (numeric):")
for i, ev in enumerate(eigenvalues_num, 1):
    print(f"  λ{i} = {ev:.6f}")
    if isinstance(ev, complex):
        print(f"      Real: {ev.real:.6f}, Imag: {ev.imag:.6f}")

In [ ]:
# Full stability analysis at DFE
stability_dfe = sir.analyze_stability_full(dfe, params)

print("="*70)
print("STABILITY ANALYSIS: Disease-Free Equilibrium")
print("="*70)
print(f"\nStability: {stability_dfe['stability'].upper()}")
print(f"Classification: {stability_dfe['classification']}")
print(f"\nEigenvalue Spectrum:")
print(f"  Max real part: {stability_dfe['max_real_part']:.6f}")
print(f"  Min real part: {stability_dfe['min_real_part']:.6f}")
print(f"  Has complex eigenvalues: {stability_dfe['has_complex']}")
print(f"  Near bifurcation: {stability_dfe['near_bifurcation']}")
if stability_dfe['bifurcation_type']:
    print(f"  Bifurcation type: {stability_dfe['bifurcation_type']}")

print(f"\nInterpretation:")
if stability_dfe['stability'] == 'unstable':
    print(f"  - DFE is UNSTABLE because R0 = {R0_numeric:.2f} > 1")
    print(f"  - Disease will invade if introduced")
    print(f"  - Epidemic will occur")
else:
    print(f"  - DFE is STABLE because R0 = {R0_numeric:.2f} < 1")
    print(f"  - Disease will die out")

In [ ]:
# Stability analysis at endemic equilibrium
if endemic:
    stability_endemic = sir.analyze_stability_full(endemic, params)
    
    print("\n" + "="*70)
    print("STABILITY ANALYSIS: Endemic Equilibrium")
    print("="*70)
    print(f"\nStability: {stability_endemic['stability'].upper()}")
    print(f"Classification: {stability_endemic['classification']}")
    print(f"\nEigenvalue Spectrum:")
    print(f"  Max real part: {stability_endemic['max_real_part']:.6f}")
    print(f"  Min real part: {stability_endemic['min_real_part']:.6f}")
    
    print(f"\nInterpretation:")
    if stability_endemic['stability'] == 'stable':
        print(f"  - Endemic equilibrium is STABLE")
        print(f"  - Disease will persist at this level")
        print(f"  - System converges to this point")

### 4.1 Stability at Bifurcation Point (R0 = 1)

In [ ]:
# Test at R0 ≈ 1
params_bifurcation = {'beta': 0.1001, 'gamma': 0.1, 'N': 1000}
R0_bif = params_bifurcation['beta'] / params_bifurcation['gamma']

stability_bifurcation = sir.analyze_stability_full(dfe, params_bifurcation)

print("Stability Analysis at Bifurcation Point (R0 ≈ 1):")
print("="*70)
print(f"R0 = {R0_bif:.4f}")
print(f"Stability: {stability_bifurcation['stability']}")
print(f"Near bifurcation: {stability_bifurcation['near_bifurcation']}")
if stability_bifurcation['bifurcation_type']:
    print(f"Bifurcation type: {stability_bifurcation['bifurcation_type']}")
print("\nThis is a transcritical bifurcation point where:")
print("  - DFE changes from stable (R0 < 1) to unstable (R0 > 1)")
print("  - Endemic equilibrium appears/disappears")

## 5. Sensitivity Analysis

Analyze how parameters affect model outputs.

In [ ]:
# Compute sensitivity matrix
print("Computing sensitivity matrix...")
sensitivity_matrix = sir.compute_sensitivity_matrix(
    output_vars=['S', 'I', 'R'],
    params=['beta', 'gamma']
)

print("\nSensitivity Matrix (symbolic):")
print("∂(output)/∂(parameter)")
print("-" * 70)

for output_var, sensitivities in sensitivity_matrix.items():
    print(f"\n{output_var}:")
    for param, sens_expr in sensitivities.items():
        print(f"  ∂{output_var}/∂{param} = {sens_expr}")

In [ ]:
# Compute elasticity indices
params_analysis = {'beta': 0.3, 'gamma': 0.1, 'N': 1000}

elasticity = sir.compute_elasticity_indices(
    params_analysis,
    output_vars=['I']
)

print("Elasticity Indices (at endemic equilibrium):")
print("="*70)
print("Interpretation: % change in output per 1% change in parameter")
print("-" * 70)

for output_var, elasticities in elasticity.items():
    print(f"\n{output_var}:")
    for param, elast_val in elasticities.items():
        print(f"  E[{output_var}, {param}] = {elast_val:.4f}")
        print(f"    → 1% increase in {param} → {elast_val:.2f}% change in {output_var}")

In [ ]:
# Perform perturbation analysis
dfe_numeric = {'S': 1000, 'I': 0, 'R': 0}

perturbation_result = sir.perform_perturbation_analysis(
    params_analysis,
    dfe_numeric,
    perturbation=0.01,  # 1% perturbation
    output_vars=['S', 'I', 'R']
)

print("Perturbation Analysis (1% parameter changes):")
print("="*70)
print("-" * 70)

for output_var, perturbations in perturbation_result.items():
    print(f"\n{output_var}:")
    for param, percent_change in perturbations.items():
        print(f"  {param}: {percent_change:+.4f}%")

In [ ]:
# Parameter importance ranking
ranking = sir.rank_parameter_importance(
    params_analysis,
    output_var='I',
    method='elasticity'
)

print("Parameter Importance Ranking for I (infectious):")
print("="*70)
print("-" * 70)
print(f"{'Rank':<6} {'Parameter':<15} {'Importance':<15} {'Interpretation'}")
print("-" * 70)

for rank, (param, importance) in enumerate(ranking, 1):
    abs_importance = abs(importance)
    if abs_importance > 0.5:
        interp = "Very important"
    elif abs_importance > 0.1:
        interp = "Important"
    elif abs_importance > 0.01:
        interp = "Moderate"
    else:
        interp = "Minor"
    
    print(f"{rank:<6} {param:<15} {importance:>8.4f}        {interp}")

## 6. Visualization

In [ ]:
# Visualize R0 as function of beta and gamma
beta_range = np.linspace(0.05, 0.5, 50)
gamma_range = np.linspace(0.05, 0.3, 50)

BETA, GAMMA = np.meshgrid(beta_range, gamma_range)
R0_grid = BETA / GAMMA

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: R0 contour
ax1 = axes[0]
contour = ax1.contour(BETA, GAMMA, R0_grid, levels=[0.5, 1, 2, 3, 5, 10], colors='black')
ax1.clabel(contour, inline=True, fontsize=10, fmt='R0=%.1f')
ax1.contourf(BETA, GAMMA, R0_grid, levels=20, cmap='RdYlGn_r')
ax1.axhline(0.1, color='blue', linestyle='--', label='γ=0.1')
ax1.axvline(0.3, color='red', linestyle='--', label='β=0.3')
ax1.plot(0.3, 0.1, 'ko', markersize=10, label='Current params')
ax1.set_xlabel('β (transmission rate)', fontsize=12)
ax1.set_ylabel('γ (recovery rate)', fontsize=12)
ax1.set_title('Basic Reproduction Number R0', fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.colorbar(ax1.collections[0], ax=ax1, label='R0')

# Plot 2: Stability regions
ax2 = axes[1]
stability = np.where(R0_grid > 1, 1, 0)  # 1 = unstable DFE, 0 = stable DFE
ax2.contourf(BETA, GAMMA, stability, levels=1, colors=['green', 'red'], alpha=0.3)
ax2.contour(BETA, GAMMA, R0_grid, levels=[1], colors='black', linewidths=2)
ax2.text(0.15, 0.15, 'DFE Stable\n(R0 < 1)', fontsize=12, ha='center')
ax2.text(0.35, 0.25, 'DFE Unstable\n(R0 > 1)', fontsize=12, ha='center')
ax2.plot(0.3, 0.1, 'ko', markersize=10, label='Current params')
ax2.set_xlabel('β (transmission rate)', fontsize=12)
ax2.set_ylabel('γ (recovery rate)', fontsize=12)
ax2.set_title('Stability Regions for Disease-Free Equilibrium', fontsize=14)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('parameter_space_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Parameter space visualization saved to 'parameter_space_analysis.png'")

In [ ]:
# Visualize endemic equilibrium as function of R0
R0_values = np.linspace(1.01, 5, 50)
N = 1000

S_star = N / R0_values
I_star = N * (1 - 1/R0_values)
R_star = np.zeros_like(R0_values)  # For SIR, R* = 0 at endemic eq

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(R0_values, S_star, 'b-', linewidth=2, label='S* (susceptible)')
ax.plot(R0_values, I_star, 'r-', linewidth=2, label='I* (infectious)')
ax.axvline(1, color='black', linestyle='--', linewidth=1.5, label='R0=1 (bifurcation)')
ax.fill_between(R0_values, 0, N, where=R0_values<1, alpha=0.2, color='green', label='DFE stable region')

ax.set_xlabel('Basic Reproduction Number (R0)', fontsize=12)
ax.set_ylabel('Population at Endemic Equilibrium', fontsize=12)
ax.set_title('Endemic Equilibrium vs R0', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 5)
ax.set_ylim(0, N)

plt.tight_layout()
plt.savefig('endemic_equilibrium_vs_R0.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Endemic equilibrium visualization saved to 'endemic_equilibrium_vs_R0.png'")

## 7. Complete Analysis Pipeline

In [ ]:
def complete_analysis(model, params, model_name="Model"):
    """
    Perform complete analysis of an epidemiological model.
    
    Args:
        model: SymbolicModel instance
        params: Parameter values dict
        model_name: Name for display
    """
    print("\n" + "="*70)
    print(f"COMPLETE ANALYSIS: {model_name}")
    print("="*70)
    
    # 1. Compute R0
    print("\n1. BASIC REPRODUCTION NUMBER")
    print("-" * 70)
    R0 = model.compute_R0_next_generation()
    R0_val = float(model.substitute_values(R0, params))
    print(f"   R0 = {R0}")
    print(f"   R0 (numeric) = {R0_val:.4f}")
    
    # 2. Find equilibria
    print("\n2. EQUILIBRIUM ANALYSIS")
    print("-" * 70)
    equilibria = model.find_all_equilibria(params)
    print(f"   Found {len(equilibria)} equilibrium point(s)")
    for i, eq in enumerate(equilibria, 1):
        print(f"\n   Equilibrium {i} ({eq['type']}):")
        for var in model.variables:
            val = eq.get(var, 0)
            if hasattr(val, 'evalf'):
                val = float(val.evalf())
            print(f"     {var}* = {val:.2f}")
    
    # 3. Stability analysis
    print("\n3. STABILITY ANALYSIS")
    print("-" * 70)
    for i, eq in enumerate(equilibria, 1):
        stability = model.analyze_stability_full(eq, params)
        print(f"\n   Equilibrium {i} ({eq['type']}):")
        print(f"     Stability: {stability['stability']}")
        print(f"     Classification: {stability['classification']}")
        print(f"     Max Re(λ): {stability['max_real_part']:.6f}")
        if stability['near_bifurcation']:
            print(f"     Near bifurcation: {stability['bifurcation_type']}")
    
    # 4. Sensitivity analysis
    print("\n4. SENSITIVITY ANALYSIS")
    print("-" * 70)
    infected_vars = model._identify_infected_compartments()
    if infected_vars:
        target_var = infected_vars[0]
        ranking = model.rank_parameter_importance(params, target_var)
        print(f"   Parameter importance ranking for {target_var}:")
        for rank, (param, importance) in enumerate(ranking, 1):
            print(f"     {rank}. {param}: {importance:.4f}")
    
    print("\n" + "="*70)
    
    return {
        'R0': R0_val,
        'equilibria': equilibria,
        'stability': [model.analyze_stability_full(eq, params) for eq in equilibria]
    }

# Run complete analysis
results = complete_analysis(sir, params, "SIR Model")

## 8. SEIR Model Example

In [ ]:
# Create SEIR model
seir = SymbolicModel()

seir.add_parameter("beta", positive=True)
seir.add_parameter("gamma", positive=True)
seir.add_parameter("epsilon", positive=True)

seir.add_variable("S", positive=True)
seir.add_variable("E", positive=True)
seir.add_variable("I", positive=True)
seir.add_variable("R", positive=True)

seir.set_total_population("N")

seir.define_ode("S", "-beta*S*I/N")
seir.define_ode("E", "beta*S*I/N - epsilon*E")
seir.define_ode("I", "epsilon*E - gamma*I")
seir.define_ode("R", "gamma*I")

print("Created SEIR model")

# Run analysis
seir_params = {'beta': 0.3, 'gamma': 0.1, 'epsilon': 0.2, 'N': 1000}
seir_results = complete_analysis(seir, seir_params, "SEIR Model")

## Summary

This notebook demonstrated:

1. **Equilibrium Finding**
   - Disease-free equilibrium (DFE)
   - Endemic equilibrium
   - Multiple equilibria detection

2. **Stability Analysis**
   - Jacobian matrix computation
   - Eigenvalue analysis
   - Stability classification (stable/unstable/saddle)
   - Bifurcation detection

3. **Sensitivity Analysis**
   - Sensitivity matrix
   - Elasticity indices
   - Perturbation analysis
   - Parameter importance ranking

4. **Visualization**
   - R0 contours
   - Stability regions
   - Endemic equilibrium curves

5. **Complete Analysis Pipeline**
   - Automated comprehensive analysis
   - Works for any compartmental model

### Key Insights

- R0 determines stability of DFE
- Bifurcation occurs at R0 = 1
- Parameter sensitivities identify control targets
- Symbolic + numeric analysis provides complete understanding